## Inverting Neural Radiance Fields

In [ ]:
import io
import os
import random

import imageio
import numpy as np
import torch
from IPython.display import display, Image
from PIL import Image as PILImage
from tqdm import tqdm

import torchutils

from models import PositionalEmbeddings, NeRF
from utils.render_utils import get_rays, render, render_path

In [ ]:
torchutils.set_seed(123456)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
trans_t = lambda t : torch.tensor([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 1, t],
    [0, 0, 0, 1]]).float()

rot_phi = lambda phi : torch.tensor([
    [1,           0,            0, 0],
    [0, np.cos(phi), -np.sin(phi), 0],
    [0, np.sin(phi),  np.cos(phi), 0],
    [0,           0,            0, 1]]).float()

rot_theta = lambda th : torch.tensor([
    [np.cos(th), 0, -np.sin(th), 0],
    [0         , 1,           0, 0],
    [np.sin(th), 0,  np.cos(th), 0],
    [0         , 0,           0, 1]]).float()

to_nerf = lambda c2w : torch.tensor([
    [-1, 0, 0, 0],
    [ 0, 0, 1, 0],
    [ 0, 1, 0, 0],
    [ 0, 0, 0, 1],
]).float() @ c2w

orbit_to_nerf = torch.tensor([
    [-1, 0, 0, 0],
    [ 0, 0, 1, 0],
    [ 0, 1, 0, 0],
    [ 0, 0, 0, 1]
]).to(device, dtype=torch.float32)

In [ ]:
def load_checkpoint(checkpoint_path, device="cuda"):
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)

    embed_pts = PositionalEmbeddings(
        **checkpoint["embed_pts"],
        periodic_functions=(torch.sin, torch.cos),
    )

    embed_viewdirs = None
    if checkpoint["embed_viewdirs"] is not None:
        embed_viewdirs = PositionalEmbeddings(
            **checkpoint["embed_viewdirs"],
            periodic_functions=(torch.sin, torch.cos),
        )

    nerf_coarse_cfg = checkpoint["nerf_coarse"]
    nerf_coarse = NeRF(
        depth=nerf_coarse_cfg["depth"],
        W=nerf_coarse_cfg["W"],
        in_channels=nerf_coarse_cfg["in_channels"],
        in_channel_views=nerf_coarse_cfg["in_channel_views"],
        out_channels=nerf_coarse_cfg["out_channels"],
        skip_connections=nerf_coarse_cfg["skip_connections"],
        use_viewdirs=nerf_coarse_cfg["use_viewdirs"],
    )
    nerf_coarse.load_state_dict(nerf_coarse_cfg["weights"])
    nerf_coarse = nerf_coarse.to(device)
    nerf_coarse.eval()

    nerf_fine = None
    if checkpoint["nerf_fine"] is not None:
        nerf_fine_cfg = checkpoint["nerf_fine"]
        nerf_fine = NeRF(
            depth=nerf_fine_cfg["depth"],
            W=nerf_fine_cfg["W"],
            in_channels=nerf_fine_cfg["in_channels"],
            in_channel_views=nerf_fine_cfg["in_channel_views"],
            out_channels=nerf_fine_cfg["out_channels"],
            skip_connections=nerf_fine_cfg["skip_connections"],
            use_viewdirs=nerf_fine_cfg["use_viewdirs"],
        )
        nerf_fine.load_state_dict(nerf_fine_cfg["weights"])
        nerf_fine = nerf_fine.to(device)
        nerf_fine.eval()

    return {
        "nerf_coarse": nerf_coarse,
        "nerf_fine": nerf_fine,
        "embed_pts": embed_pts,
        "embed_viewdirs": embed_viewdirs,
        "n_samples": checkpoint["n_samples"],
        "n_importance": checkpoint["n_importance"],
        "batch_size": checkpoint["batch_size"],
        "hwf": checkpoint["hwf"],
        "K": checkpoint["K"].to(device),
        "chunk": checkpoint["chunk"],
        "near": checkpoint["near"],
        "far": checkpoint["far"],
        "use_white_background": checkpoint["use_white_background"],
    }

In [ ]:
def camera_pose_from_orbit(theta: float, phi: float, radius: float = 4.0) -> torch.Tensor:
    zero = theta * 0.0
    one = zero + 1.0
    radius_tensor = zero + radius
    translation = torch.stack([
        torch.stack([one, zero, zero, zero]),
        torch.stack([zero, one, zero, zero]),
        torch.stack([zero, zero, one, radius_tensor]),
        torch.stack([zero, zero, zero, one]),
    ])
    cos_phi, sin_phi = torch.cos(phi), torch.sin(phi)
    phi_rotation = torch.stack([
        torch.stack([one, zero, zero, zero]),
        torch.stack([zero, cos_phi, -sin_phi, zero]),
        torch.stack([zero, sin_phi, cos_phi, zero]),
        torch.stack([zero, zero, zero, one]),
    ])
    cos_theta, sin_theta = torch.cos(theta), torch.sin(theta)
    theta_rotation = torch.stack([
        torch.stack([cos_theta, zero, -sin_theta, zero]),
        torch.stack([zero, one, zero, zero]),
        torch.stack([sin_theta, zero, cos_theta, zero]),
        torch.stack([zero, zero, zero, one]),
    ])
    return orbit_to_nerf @ theta_rotation @ phi_rotation @ translation

def step(target_pixels: torch.Tensor, theta: torch.Tensor, phi: torch.Tensor, optimizer: torch.optim.Optimizer) -> float:
    optimizer.zero_grad()
    pose = camera_pose_from_orbit(theta, phi)
    rays_origin, rays_direction = get_rays(
        height,
        width,
        lego_nerf["K"],
        pose[:3, :4],
    )
    rays = torch.stack([
        rays_origin.reshape(-1, 3)[ray_indices],
        rays_direction.reshape(-1, 3)[ray_indices],
    ])
    output = render(
        nerf_coarse=lego_nerf["nerf_coarse"],
        nerf_fine=lego_nerf["nerf_fine"],
        embed_pts=lego_nerf["embed_pts"],
        embed_viewdirs=lego_nerf["embed_viewdirs"],
        rays=rays,
        n_samples=lego_nerf["n_samples"],
        n_importance=lego_nerf["n_importance"],
        chunk=lego_nerf["chunk"],
        near=lego_nerf["near"],
        far=lego_nerf["far"],
        use_white_background=lego_nerf["use_white_background"],
    )
    prediction = output.get("rgb_map_fine", output["rgb_map"])
    loss = ((prediction - target_pixels) ** 2).mean()
    loss.backward()
    optimizer.step()
    return loss

### Lego

In [ ]:
lego_nerf_checkpoint_path = os.path.join("checkpoints", "lego", "lego_200000.pth")
lego_nerf = load_checkpoint(lego_nerf_checkpoint_path, device=device)

In [ ]:
radius = 4.0
phi = random.uniform(-90.0, 0.0)

theta = random.uniform(-180.0, 180.0)
c2w = trans_t(radius)
c2w = rot_phi(phi/180. * np.pi) @ c2w
c2w = rot_theta(theta/180. * np.pi) @ c2w
c2w = to_nerf(c2w)
render_pose = c2w.to(device)
render_pose = render_pose.unsqueeze(0)

In [ ]:
rgbs = render_path(
    nerf_coarse=lego_nerf["nerf_coarse"],
	nerf_fine=lego_nerf["nerf_fine"],
	embed_pts=lego_nerf["embed_pts"],
	embed_viewdirs=lego_nerf["embed_viewdirs"],
	render_poses=render_pose,
	n_samples=lego_nerf["n_samples"],
	n_importance=lego_nerf["n_importance"],
	batch_size=lego_nerf["batch_size"],
	hwf=lego_nerf["hwf"],
	K=lego_nerf["K"],
	chunk=lego_nerf["chunk"],
	near=lego_nerf["near"],
	far=lego_nerf["far"],
	use_white_background=lego_nerf["use_white_background"],
)

In [ ]:
img = rgbs[0]
img_uint8 = (img * 255).astype("uint8")

buf = io.BytesIO()
PILImage.fromarray(img_uint8).save(buf, format="PNG")
buf.seek(0)

display(Image(data=buf.getvalue()))

In [ ]:
candidates = {
    "radius": radius,
    "thetas": [0.0, 90.0, 180.0, 270.0, 0.0],
    "phis": [0.0, 0.0, 0.0, 0.0, -90.0],
    "names": ["Front", "Right", "Back", "Left", "Top"],
    "poses": [],
}

for theta, phi in zip(candidates["thetas"], candidates["phis"]):
    pose = trans_t(candidates["radius"])
    pose = rot_phi(np.deg2rad(phi)) @ pose
    pose = rot_theta(np.deg2rad(theta)) @ pose
    candidates["poses"].append(to_nerf(pose))

candidates["poses"] = torch.stack(candidates["poses"]).to(device)

In [ ]:
render_kwargs = dict(
    nerf_coarse=lego_nerf["nerf_coarse"],
    nerf_fine=lego_nerf["nerf_fine"],
    embed_pts=lego_nerf["embed_pts"],
    embed_viewdirs=lego_nerf["embed_viewdirs"],
    n_samples=lego_nerf["n_samples"],
    n_importance=lego_nerf["n_importance"],
    batch_size=lego_nerf["batch_size"],
    hwf=lego_nerf["hwf"],
    K=lego_nerf["K"],
    chunk=lego_nerf["chunk"],
    near=lego_nerf["near"],
    far=lego_nerf["far"],
    use_white_background=lego_nerf["use_white_background"],
)

In [ ]:
target_rgb = rgbs[0]
target_tensor = torch.from_numpy(target_rgb).to(device)
height, width, _ = lego_nerf["hwf"]
ray_indices = torch.randperm(height * width, device=device)[:1024]
target_pixels = target_tensor.reshape(-1, 3)[ray_indices]

target_orbit_pose = orbit_to_nerf.T @ render_pose[0]
target_position = target_orbit_pose[:3, 3]
target_theta = np.rad2deg(np.arctan2(-target_position[0].item(), target_position[2].item()))
target_phi = np.rad2deg(np.arcsin(np.clip(-target_position[1].item() / radius, -1.0, 1.0)))

candidate_states = []
for name, candidate_theta, candidate_phi in zip(candidates["names"], candidates["thetas"], candidates["phis"]):
    theta_parameter = torch.nn.Parameter(
        torch.tensor(np.deg2rad(candidate_theta), dtype=torch.float32, device=device)
    )
    phi_parameter = torch.nn.Parameter(
        torch.tensor(np.deg2rad(candidate_phi), dtype=torch.float32, device=device)
    )
    optimizer = torch.optim.Adam([theta_parameter, phi_parameter], lr=0.01)

    losses = []
    pbar = tqdm(range(10), desc=name)
    for _ in pbar:
        loss = step(target_pixels, theta_parameter, phi_parameter, optimizer)
        losses.append(loss.item())
        psnr = -10.0 * np.log10(loss.item())
        pbar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "psnr": f"{psnr.item():.2f}",
        })

    candidate_states.append({
        "theta": theta_parameter,
        "phi": phi_parameter,
        "optimizer": optimizer,
        "losses": losses,
    })

poses = torch.stack([
    camera_pose_from_orbit(state["theta"].detach(), state["phi"].detach())
    for state in candidate_states
])
rgbs = render_path(render_poses=poses, **render_kwargs)
mses = ((rgbs - target_rgb[None, ...]) ** 2).mean(axis=(1, 2, 3))
psnr = -10.0 * np.log10(mses)
idx = int(np.argmin(mses))

best_state = candidate_states[idx]
trajectory_poses = []

pbar = tqdm(range(100))
for _ in pbar:
    loss = step(target_pixels, best_state["theta"], best_state["phi"], best_state["optimizer"])
    psnr = -10.0 * np.log10(loss.item())
    pbar.set_postfix({
		"loss": f"{loss.item():.4f}",
		"psnr": f"{psnr.item():.2f}",
	})
    trajectory_poses.append(
        camera_pose_from_orbit(
            best_state["theta"].detach(), best_state["phi"].detach()
        ).detach()
    )

trajectory_poses = torch.stack(trajectory_poses)
pose = trajectory_poses[-1]

final_rgb = render_path(render_poses=pose.unsqueeze(0), **render_kwargs)[0]
final_mse = ((final_rgb - target_rgb) ** 2).mean()
final_psnr = -10.0 * np.log10(final_mse)
print(f"Final PSNR: {final_psnr:.2f}")

gif_path = "lego.gif"
gif_opacity = 0.5
gif_frame_count = len(trajectory_poses) + 1
with imageio.get_writer(gif_path, mode="I", duration=0.08, loop=0) as gif:
    gif.append_data(np.clip(target_rgb * 255.0, 0.0, 255.0).astype(np.uint8))
    for trajectory_pose in tqdm(trajectory_poses, total=len(trajectory_poses)):
        trajectory_rgb = render_path(
            render_poses=trajectory_pose.unsqueeze(0),
            **render_kwargs,
        )[0]
        blended_rgb = (1.0 - gif_opacity) * target_rgb + gif_opacity * trajectory_rgb
        gif.append_data(np.clip(blended_rgb * 255.0, 0.0, 255.0).astype(np.uint8))